In [ ]:
import geopandas as gp
import pandas as pd
import os
import numpy as np
import re
from collections import Counter
import pber_functions_v1 as pber

# Alabama 2024 General Election Results

In [ ]:
results = pd.read_csv("./raw-from-source/al_2024_gen_prec_csv/al_2024_gen_prec_csv.csv")

In [ ]:
jefferson = results[results["County"]=="Jefferson"].copy()
others = results[results["County"]!="Jefferson"].copy()
races = [i for i in list(results.columns) if i not in ["UNIQUE_ID","COUNTYFP","County","Precinct"]]

In [ ]:
jefferson_absentee = jefferson[jefferson["Precinct"].str.contains("ABSENTEE") | jefferson["Precinct"].str.contains("PROVISIONAL")].copy(deep = True)

In [ ]:
jefferson_absentee

In [ ]:
jefferson_absentee_divs = jefferson_absentee[jefferson_absentee["Precinct"].str.contains("BESSEMER") | jefferson_absentee["Precinct"].str.contains("BIRMINGHAM")].copy(deep = True)
jefferson_absentee_cnty = jefferson_absentee[~jefferson_absentee["Precinct"].str.contains("BESSEMER") & ~jefferson_absentee["Precinct"].str.contains("BIRMINGHAM")].copy(deep = True)


In [ ]:
jefferson_absentee_divs

In [ ]:
jefferson_absentee_divs_bess = jefferson_absentee_divs[jefferson_absentee_divs["Precinct"].str.contains("BESSEMER")].copy(deep = True)
jefferson_absentee_divs_birm = jefferson_absentee_divs[jefferson_absentee_divs["Precinct"].str.contains("BIRMINGHAM")].copy(deep = True)

In [ ]:
jefferson_allocate = jefferson[~(jefferson["Precinct"].str.contains("ABSENTEE")) & ~(jefferson["Precinct"].str.contains("PROVISIONAL"))].copy(deep = True)

In [ ]:
#jefferson_allocate[["UNIQUE_ID","County"]].to_csv("./jefferson_precs.csv", index = False)

In [ ]:
jefferson_divisions = pd.read_csv("./raw-from-source/jefferson_divisions.csv")
jefferson_divisions.drop(["County"], axis = 1, inplace = True)

In [ ]:
jefferson_divisions["Division"].value_counts()

In [ ]:
jefferson_divisions["Division2"] = np.where(jefferson_divisions["Division"]=="BOTH","BIRMINGHAM",jefferson_divisions["Division"])
jefferson_divisions["Division1"] = np.where(jefferson_divisions["Division"]=="BOTH","BESSEMER",jefferson_divisions["Division"])

In [ ]:
jefferson_allocate_full = pd.merge(jefferson_allocate,jefferson_divisions,on="UNIQUE_ID", how = "outer", indicator = True)

In [ ]:
birmingham_precs = jefferson_allocate_full[jefferson_allocate_full["Division2"]=="BIRMINGHAM"].copy(deep = True)
bessemer_precs = jefferson_allocate_full[jefferson_allocate_full["Division1"]=="BESSEMER"].copy(deep = True)

In [ ]:
birmingham_precs_allocated = pber.allocate_absentee(birmingham_precs, jefferson_absentee_divs_birm, races, "COUNTYFP")
bessemer_precs_allocated = pber.allocate_absentee(bessemer_precs, jefferson_absentee_divs_bess, races, "COUNTYFP")

In [ ]:
for prec in set(birmingham_precs_allocated[birmingham_precs_allocated["Division"]=="BOTH"]["UNIQUE_ID"]):
    for race in races:
        print(birmingham_precs_allocated.loc[birmingham_precs_allocated["UNIQUE_ID"]==prec,race].values[0])
        print(results.loc[results["UNIQUE_ID"]==prec,race].values[0])
        birmingham_precs_allocated.loc[birmingham_precs_allocated["UNIQUE_ID"]==prec,race] -= results.loc[results["UNIQUE_ID"]==prec,race].values[0]
        print(birmingham_precs_allocated.loc[birmingham_precs_allocated["UNIQUE_ID"]==prec,race].values[0])
        print("")

In [ ]:
jefferson_final = pd.concat([bessemer_precs_allocated,birmingham_precs_allocated])

In [ ]:
jefferson_final.drop(["Division","Division2","Division1","_merge"], axis = 1, inplace = True)

In [ ]:
jefferson_final[jefferson_final["G24A01NO"].isna()]

In [ ]:
for race in races:
    print(race)
    jefferson_final[race] = jefferson_final[race].astype(int)

In [ ]:
jefferson_final = jefferson_final.groupby(["UNIQUE_ID","COUNTYFP","County","Precinct"], as_index = False).sum()

In [ ]:
others = pd.concat([others, jefferson_final, jefferson_absentee_cnty])

In [ ]:
others.reset_index(inplace = True, drop = True)

In [ ]:
others_allocate = others[~(others["Precinct"].str.contains("ABSENTEE")) & ~(others["Precinct"].str.contains("PROVISIONAL")) & ~(others["Precinct"].str.contains("PROVSIONAL"))].copy(deep = True)

In [ ]:
others_absentee = others[(others["Precinct"].str.contains("ABSENTEE")) | (others["Precinct"].str.contains("PROVISIONAL")) | (others["Precinct"].str.contains("PROVSIONAL"))].copy(deep = True)

In [ ]:
others_allocated = pber.allocate_absentee(others_allocate, others_absentee, races, "COUNTYFP")

In [ ]:
others_allocated

In [ ]:
for race in races:
    others_allocated[race] = others_allocated[race].astype(int)
    results[race] = results[race].astype(int)
    print(sum(others_allocated[race])-sum(results[race]))

In [ ]:
pber.county_totals_check(results, "Pre-Alloc", others_allocated, "Post-Alloc", races, "County", full_print=False, method='county')

In [ ]:
set(others_allocated["Precinct"].unique())

In [ ]:
others_allocated["COUNTYFP"] = others_allocated["COUNTYFP"].astype(str).str.zfill(3)

In [ ]:
others_allocated["UNIQUE_ID"] = others_allocated["County"] + "-:-" + others_allocated["Precinct"]

In [ ]:
others_allocated.to_csv("./al_2024_gen_prec_allocated.csv", index = False)